# 10 — Advanced Indexing (Parent-Doc, Multi-Rep, Hybrid BM25, RAPTOR & ColBERT)

**By the end of this notebook, you can decouple search vectors from delivered context via Parent-Doc/Multi-Rep indexing, fuse BM25+Dense hybrid search, and run demo-scale RAPTOR & ColBERT MaxSim.**

### Learning Objectives
- Implement Parent-Document and Multi-Representation Summary indexing cleanly with InMemoryVectorStore.
- Compare Dense-only failure on exact error tokens vs Hybrid (BM25 + Dense RRF) retrieval.
- Build honest, demo-scale implementations of RAPTOR hierarchical summaries and ColBERT token-level MaxSim.

### Prerequisites
- Notebooks 00–09

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 01:29:20.566000 18728 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Decoupling What You Index from What You Deliver

Standard indexing embeds every chunk identically. **Advanced Indexing** breaks that limit in four ways:
1. **Parent-Document & Multi-Representation**: Embed short child chunks (`130` chars) or concise summaries for crisp matching, then deliver the full `650`-char parent passage!
2. **Hybrid (BM25 + Dense RRF)**: Pairs exact word matching (`ERR_QUANTUM_TIMEOUT`) with dense vectors (`all-MiniLM-L6-v2`).
3. **RAPTOR (Demo Scale)**: Recursively clusters & summarizes chunks into a multi-level summary tree for high-level overview queries.
4. **ColBERT Late Interaction (Demo Scale)**: Stores one vector per *token* rather than compressing a paragraph into one vector, scoring query vs doc via **MaxSim**. *(Full ColBERT/RAPTOR scale requires GPU/specialized indexes in production.)*

```
Query Tokens [q1, q2] --MaxSim--> Doc Tokens [d1..dm]   (ColBERT Late Interaction)

Leaf Chunks --Cluster--> Cluster Summaries --> Root Tree Index (RAPTOR Demo)
Tiny Child Chunk (precision search) --parent_id--> Full Parent Block (rich context!)
```

## 3. Step-by-Step Build

### Step 1: Parent-Document Indexing & Hydration
Let's link tiny 130-char child snippets via `parent_id` inside `InMemoryVectorStore` back to full 650-char parent blocks.

In [2]:
import uuid
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_corpus_documents

parents = RecursiveCharacterTextSplitter(chunk_size=650, chunk_overlap=80).split_documents(load_corpus_documents(data_dir))
parent_map, child_docs = {}, []
child_split = RecursiveCharacterTextSplitter(chunk_size=130, chunk_overlap=20)
for p in parents:
    pid = str(uuid.uuid4())[:8]; parent_map[pid] = p
    for c in child_split.split_documents([p]): c.metadata["parent_id"] = pid; child_docs.append(c)
child_idx = InMemoryVectorStore.from_documents(child_docs, embeddings)
top_pid = child_idx.similarity_search("ZeroGravity 40 milliseconds threshold", k=1)[0].metadata["parent_id"]
print(f"Matched tiny child (~130c) -> Hydrated Parent ({len(parent_map[top_pid].page_content)}c):", parent_map[top_pid].metadata["source"])
        

Matched tiny child (~130c) -> Hydrated Parent (437c): product_faq.md


> **What just happened?**
> Tiny chunks maximize needle-in-a-haystack search while parents preserve surrounding tier prices and caveats.

### Step 2: Multi-Representation Summary Index
Summarize each source document upfront and index the clean summaries pointing back to complete raw files.

In [3]:
from langchain_core.documents import Document
raw_docs = load_corpus_documents(data_dir)
sum_docs = [Document(page_content=llm.invoke(f"Write a concise summary:\nDOCUMENT:\n{d.page_content[:1800]}").content, metadata={"source": d.metadata["source"]}) for d in raw_docs]
sum_index = InMemoryVectorStore.from_documents(sum_docs, embeddings)
top_sum_src = sum_index.similarity_search("ROS2 Linux distributions", k=1)[0].metadata["source"]
print("Summary index routed 'ROS2 Linux distributions' ->", top_sum_src)
assert top_sum_src == "product_faq.md"
        

Summary index routed 'ROS2 Linux distributions' -> product_faq.md


> **What just happened?**
> Multi-representation abstracts away noisy formatting or boilerplate logs during vector lookup.

### Step 3: Failure Case vs. Hybrid Search (BM25 + Dense RRF)
On exact technical error tokens like `9872 ERR_QUANTUM_TIMEOUT`, pure dense vectors sometimes let general topics creep into top results; **Hybrid BM25+Dense RRF** locks onto exact identifiers cleanly!

In [4]:
import re
from rank_bm25 import BM25Okapi
tok = lambda s: re.findall(r"[a-zA-Z0-9_]+", s.lower())
dense_idx = InMemoryVectorStore.from_documents(parents, embeddings)
bm25 = BM25Okapi([tok(p.page_content) for p in parents])
q_err = "ERR_QUANTUM_TIMEOUT 9872 systemctl"
bm_scores = bm25.get_scores(tok(q_err))
bm_top = [parents[i] for i in sorted(range(len(parents)), key=lambda i: bm_scores[i], reverse=True)[:3]]
rrf = {}
for lst in (dense_idx.similarity_search(q_err, k=3), bm_top):
    for r, d in enumerate(lst, 1): rrf[d.page_content] = rrf.get(d.page_content, 0.0) + 1.0 / (60 + r)
best_txt = max(rrf, key=rrf.get)
print("Hybrid #1 contains exact command & port?:", "9872" in best_txt and "systemctl" in best_txt)
        

Hybrid #1 contains exact command & port?: True


> **What just happened?**
> Combining lexical BM25 exactness with semantic embeddings improves technical troubleshooting accuracy over NB05 dense baseline.

### Step 4: Demo-Scale RAPTOR Hierarchical Summary Tree
Note: Full RAPTOR runs UMAP + GMM over thousands of chunks. Here is an honest **demo-scale RAPTOR** clustering leaf chunks with `KMeans(n_clusters=2)`, summarizing each cluster, and indexing both levels!

In [5]:
import numpy as np
from sklearn.cluster import KMeans
leaf_vecs = np.array(embeddings.embed_documents([p.page_content for p in parents]))
labels = KMeans(n_clusters=2, n_init=5, random_state=42).fit_predict(leaf_vecs)
cluster_docs = []
for cid in (0, 1):
    joined = "\n".join(p.page_content[:250] for p, lbl in zip(parents, labels) if lbl == cid)
    summ = llm.invoke(f"Write a concise summary of cluster {cid}:\nDOCUMENT:\n{joined[:800]}").content
    cluster_docs.append(Document(page_content=summ, metadata={"level": "raptor_cluster", "cluster": cid}))
raptor_store = InMemoryVectorStore.from_documents(parents + cluster_docs, embeddings)
hit = raptor_store.similarity_search("high level cluster overview summary", k=1)[0]
print("RAPTOR level:", hit.metadata.get("level", "leaf"), "| preview:", hit.page_content[:90].replace("\n", " "))
        

RAPTOR level: leaf | preview: # Novasphere OS — Technical Product FAQ (2026)  ## Service Level Agreements (SLA) & Reliab


> **What just happened?**
> Adding higher-level cluster summaries lets broad 'theme' queries retrieve holistic answers cleanly alongside fine details.

### Step 5: Demo-Scale ColBERT Late Interaction (`MaxSim`) Intuition
Note: Real ColBERT indexes 128-d token vectors with PyTorch/PLAID GPU acceleration. Here is a crisp **NumPy demo-scale MaxSim** scoring every query token against every doc token!

In [6]:
def demo_colbert_maxsim(query_str: str, doc_str: str) -> float:
    q_toks = query_str.lower().split()[:5]
    d_toks = doc_str.lower().split()[:12]
    qv = np.array(embeddings.embed_documents(q_toks))
    dv = np.array(embeddings.embed_documents(d_toks))
    sim_matrix = qv @ dv.T
    return float(sim_matrix.max(axis=1).sum())

s_faq = demo_colbert_maxsim("port 9872 timeout", "allow UDP port 9872 quantum timeout")
s_off = demo_colbert_maxsim("port 9872 timeout", "dogs allowed Zurich office hub Tuesday")
print(f"ColBERT MaxSim on target={s_faq:.3f} vs unrelated={s_off:.3f}")
assert s_faq > s_off
        

ColBERT MaxSim on target=3.000 vs unrelated=0.841


> **What just happened?**
> Late interaction preserves per-word alignment rather than blurring rare tokens inside a single average vector.

## 4. Try It Yourself
1. **Compare Parent-Child chunk counts** — *Hint:* Print `len(parents)` vs `len(child_docs)`.
2. **Verify Hybrid beats random mismatch** — *Hint:* Check `s_faq > s_off` under MaxSim and exact BM25 token score on `9872`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [7]:
print(f"Parents={len(parents)} vs Children={len(child_docs)} (ratio={len(child_docs)/len(parents):.1f}x)")
print("BM25 top score on '9872':", round(float(max(bm25.get_scores(tok("9872")))), 3))
        

Parents=16 vs Children=92 (ratio=5.8x)
BM25 top score on '9872': 2.006


## 5. Common Mistakes
- Claiming a toy cluster or 12-token loop is production ColBERT/RAPTOR — always be transparent about demo scale vs library scale.
- Omitting parent `pid` deduplication when multiple children match the same parent block.
- Blending raw unbounded BM25 scores directly with cosine similarities instead of RRF.

## 6. Recap
- Parent-Document and Multi-Rep indexing cleanly separate fine search vectors from full context.
- Hybrid BM25+Dense RRF protects exact product codes and numbers where vectors alone soften precision.
- RAPTOR and ColBERT illustrate hierarchical summarization and late token-level MaxSim concepts clearly.

## 7. Next Notebook
Next (`11_reranking_and_compression.ipynb`): Filter Stage-1 hits with a REAL local neural `CrossEncoder` reranker and compress prompts cleanly!

## 8. Video Chapter Outline
1. **Why Single-Pass Uniform Indexing Hits Limits** — Overview of 4 indexing strategies.
2. **Parent-Doc & Multi-Rep Summary Indexes** — Small search keys + rich context.
3. **Hybrid BM25 + Dense RRF** — Solving exact error-token lookup.
4. **RAPTOR Hierarchical Tree (Demo Scale)** — Clustering + summarizing chunks cleanly.
5. **ColBERT Token MaxSim (Demo Scale)** — Token-matrix matching intuition.